But : features morphologiques automatiques (longueurs, diamètres, #branches, Sholl, proportion axone/dendrite, etc.), puis features.csv.

In [ ]:
import neurom as nm
from neurom import features as F
import pandas as pd
from pathlib import Path

def feats(n):
    return {
        "n_sections": nm.get("number_of_sections", n),
        "total_length": sum(nm.get("section_lengths", n)),
        "mean_diameter": pd.Series(nm.get("section_diameters", n)).mean(),
        "max_branch_order": max(nm.get("branch_orders", n)),
        "n_axon": sum(1 for _ in nm.iter_neurites(n, filt=lambda t: t.type==nm.core.types.NeuriteType.axon)),
        "n_dend": sum(1 for _ in nm.iter_neurites(n, filt=lambda t: t.type!=nm.core.types.NeuriteType.axon)),
        # ajoute Sholl si souhaité avec neurom.analysis.morphmath
    }

manifest = pd.read_csv("../data/interim/manifest.csv")
rows=[]
for path,label,cond in manifest[["path","morphotype","condition"]].itertuples(index=False):
    n = nm.load_neuron(path)
    d = feats(n); d.update({"path":path, "morphotype":label, "condition":cond})
    rows.append(d)
features = pd.DataFrame(rows)
features.to_csv("../data/processed/features.csv", index=False)
features.head()
